# 01 - Zillow & Census Data Cleaning

**Part 1: Zillow** — Clean ZHVI data for NY State  
**Part 2: Census** — Clean ACS 5-Year tables (B01001, B15003, B19013, B25064)

## 1. Load Libraries and Data

In [98]:
import pandas as pd

# Load Zillow NY data (county-level, 2000-2026)
df = pd.read_csv('zillow_data_ny_full.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (62, 322)


,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,31-01-2000,...,30-04-2025,31-05-2025,30-06-2025,31-07-2025,31-08-2025,30-09-2025,31-10-2025,30-11-2025,31-12-2025,31-01-2026
0,581,6,Kings County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,47,206414.8913,...,8.954248e+05,9.004064e+05,9.053890e+05,9.085524e+05,9.113245e+05,9.143632e+05,9.177690e+05,9.210083e+05,9.237443e+05,9.284700e+05
1,1347,10,Queens County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,81,246786.6268,...,7.040135e+05,7.083069e+05,7.123719e+05,7.148848e+05,7.162041e+05,7.182731e+05,7.215364e+05,7.249828e+05,7.278390e+05,7.303629e+05
2,2452,20,New York County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,61,385633.2542,...,1.199182e+06,1.192137e+06,1.185157e+06,1.181379e+06,1.179924e+06,1.182318e+06,1.187450e+06,1.192495e+06,1.196587e+06,1.203576e+06
3,2046,24,Suffolk County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,103,189425.8550,...,6.746249e+05,6.747727e+05,6.749732e+05,6.760666e+05,6.768846e+05,6.773768e+05,6.782278e+05,6.815436e+05,6.853688e+05,6.882239e+05
4,401,26,Bronx County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,5,216965.9428,...,4.769460e+05,4.790601e+05,4.810051e+05,4.828006e+05,4.839404e+05,4.853390e+05,4.867583e+05,4.886704e+05,4.890837e+05,4.912303e+05


## 2. Convert Wide → Long Format

Melt monthly columns into a single `date` column with `zhvi` values.

In [99]:
# Columns to keep as identifiers (everything except monthly ZHVI columns)
# Full file: RegionName = county (e.g., "Kings County"); has FIPS, no CountyName
id_vars = ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS']
df_long = df.melt(
    id_vars=id_vars,
    var_name='date',
    value_name='zhvi'
)
# Full file is county-level: RegionName = county name
df_long['CountyName'] = df_long['RegionName']

print(f"Long format shape: {df_long.shape}")
df_long.head(10)

Long format shape: (19406, 12)


,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,date,zhvi,CountyName
0,581,6,Kings County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,47,31-01-2000,206414.89130,Kings County
1,1347,10,Queens County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,81,31-01-2000,246786.62680,Queens County
2,2452,20,New York County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,61,31-01-2000,385633.25420,New York County
3,2046,24,Suffolk County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,103,31-01-2000,189425.85500,Suffolk County
4,401,26,Bronx County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,5,31-01-2000,216965.94280,Bronx County
5,1252,29,Nassau County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,59,31-01-2000,248424.18700,Nassau County
6,3148,48,Westchester County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,119,31-01-2000,288827.60600,Westchester County
7,157,55,Erie County,county,NY,NY,"Buffalo-Cheektowaga, NY",36,29,31-01-2000,88096.37748,Erie County
8,1223,87,Monroe County,county,NY,NY,"Rochester, NY",36,55,31-01-2000,98890.15546,Monroe County
9,2511,143,Richmond County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,85,31-01-2000,187331.82910,Richmond County


## 3. Convert Date and Extract Year

In [100]:
# Convert date string to datetime (full file uses dd-mm-yyyy)
df_long['date'] = pd.to_datetime(df_long['date'], format='%d-%m-%Y')

# Extract year for merging with yearly Census data
df_long['year'] = df_long['date'].dt.year

# Keep 2021-2026 (2021 for lagged; 2022-2024 Census; 2025-2026 for 24-25 split)
df_long = df_long[df_long['year'].isin([2021, 2022, 2023, 2024, 2025, 2026])]

print(df_long['date'].min(), '→', df_long['date'].max())
print(df_long['year'].unique())
df_long.head()

2021-01-31 00:00:00 → 2024-12-31 00:00:00
[2021 2022 2023 2024]


,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,date,zhvi,CountyName,year
15624,581,6,Kings County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,47,2021-01-31,8.095359e+05,Kings County,2021
15625,1347,10,Queens County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,81,2021-01-31,6.671342e+05,Queens County,2021
15626,2452,20,New York County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,61,2021-01-31,1.332834e+06,New York County,2021
15627,2046,24,Suffolk County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,103,2021-01-31,5.040655e+05,Suffolk County,2021
15628,401,26,Bronx County,county,NY,NY,"New York-Newark-Jersey City, NY-NJ-PA",36,5,2021-01-31,5.541779e+05,Bronx County,2021


## 4. Basic Data Checks

In [101]:
# Check for missing values
print("Missing values:")
print(df_long.isnull().sum())

# Check ZHVI range
print(f"\nZHVI range: {df_long['zhvi'].min():,.0f} - {df_long['zhvi'].max():,.0f}")

# Unique counties
print(f"\nUnique counties: {df_long['CountyName'].nunique()}")

Missing values:
RegionID               0
SizeRank               0
RegionName             0
RegionType             0
StateName              0
State                  0
Metro                480
StateCodeFIPS          0
MunicipalCodeFIPS      0
date                   0
zhvi                   0
CountyName             0
year                   0
dtype: int64

ZHVI range: 91,566 - 1,471,889

Unique counties: 62


## 5. Aggregate to County-Year Level

Census data is yearly and county-level. Aggregate Zillow by taking the **mean** of monthly ZHVI per county per year (full file is already county-level).

In [102]:
df_county_yearly = (
    df_long
    .groupby(['CountyName', 'year'])
    .agg({'zhvi': 'mean'})
    .reset_index()
)

df_county_yearly.columns = ['county', 'year', 'zhvi']
print(f"County-year shape: {df_county_yearly.shape}")
df_county_yearly.head(15)

County-year shape: (248, 3)


,county,year,zhvi
0,Albany County,2021,267142.696817
1,Albany County,2022,291524.127425
2,Albany County,2023,306901.238467
3,Albany County,2024,327621.834283
4,Allegany County,2021,98586.514358
5,Allegany County,2022,110579.108158
6,Allegany County,2023,110350.300983
7,Allegany County,2024,117169.827933
8,Bronx County,2021,550709.673433
9,Bronx County,2022,519156.283608


## 5b. Seasonality Score (School Season vs Winter)

% higher in Aug–Oct (back-to-school) vs Dec–Feb (winter). One value per county-year.

In [103]:
df_long['month'] = pd.to_datetime(df_long['date']).dt.month

school_months = [8, 9, 10]   # Aug–Oct (back-to-school)
winter_months = [1, 2, 12]   # Dec–Feb

school_avg = df_long[df_long['month'].isin(school_months)].groupby(['CountyName', 'year'])['zhvi'].mean()
winter_avg = df_long[df_long['month'].isin(winter_months)].groupby(['CountyName', 'year'])['zhvi'].mean()

seasonality = (school_avg - winter_avg) / winter_avg
seasonality_df = seasonality.reset_index()
seasonality_df.columns = ['county', 'year', 'seasonality_score']

df_county_yearly = df_county_yearly.merge(seasonality_df, on=['county', 'year'], how='left')
print(f"Seasonality range: {df_county_yearly['seasonality_score'].min():.4f} to {df_county_yearly['seasonality_score'].max():.4f}")
df_county_yearly.head()

Seasonality range: -0.0411 to 0.0967


,county,year,zhvi,seasonality_score
0,Albany County,2021,267142.696817,0.042619
1,Albany County,2022,291524.127425,0.040014
2,Albany County,2023,306901.238467,0.026890
3,Albany County,2024,327621.834283,0.026343
4,Allegany County,2021,98586.514358,0.059136


## 6. Save Cleaned Data

In [104]:
# Save long format (city-month level) - useful for detailed analysis
df_long.to_csv('zillow_long_cleaned.csv', index=False)
print(f"Saved zillow_long_cleaned.csv ({len(df_long):,} rows)")

# Save county-year aggregated - for merging with Census
df_county_yearly.to_csv('zillow_county_yearly.csv', index=False)
print(f"Saved zillow_county_yearly.csv ({len(df_county_yearly):,} rows)")

Saved zillow_long_cleaned.csv (2,976 rows)
Saved zillow_county_yearly.csv (248 rows)


## Part 1 Summary

- **zillow_long_cleaned.csv**: City-level, monthly ZHVI (for computing target_growth)
- **zillow_county_yearly.csv**: County-level, yearly mean ZHVI (ready for Census merge)

---

# Part 2: Census Data Cleaning

Load ACS 5-Year tables (skip metadata row), create CountyFIPS, extract county name for merge.

## 7. Load Census Tables

Load each ACS table with `skiprows=[1]` to skip the metadata row. Extract CountyFIPS and county name.

In [105]:
def load_census_file(path, year):
    """Load Census CSV, skip metadata row, add CountyFIPS and county name."""
    df = pd.read_csv(path, skiprows=[1])
    df['CountyFIPS'] = df['GEO_ID'].str[-5:]
    df['county'] = df['NAME'].str.replace(', New York', '')
    df['year'] = year
    return df

In [106]:
# B19013 - Median Income
income_2022 = load_census_file('median_income/ACSDT5Y2022.B19013-Data.csv', 2022)[['CountyFIPS', 'county', 'year', 'B19013_001E']]
income_2023 = load_census_file('median_income/ACSDT5Y2023.B19013-Data.csv', 2023)[['CountyFIPS', 'county', 'year', 'B19013_001E']]
income_2024 = load_census_file('median_income/ACSDT5Y2024.B19013-Data.csv', 2024)[['CountyFIPS', 'county', 'year', 'B19013_001E']]
df_income = pd.concat([income_2022, income_2023, income_2024], ignore_index=True)
df_income = df_income.rename(columns={'B19013_001E': 'median_income'})
df_income['median_income'] = pd.to_numeric(df_income['median_income'], errors='coerce')
print("Income:", df_income.shape)
df_income.head()

Income: (186, 4)


,CountyFIPS,county,year,median_income
0,36001,Albany County,2022,78829
1,36003,Allegany County,2022,58725
2,36005,Bronx County,2022,47036
3,36007,Broome County,2022,58317
4,36009,Cattaraugus County,2022,56889


In [107]:
# B25064 - Median Rent
rent_2022 = load_census_file('median_rent/ACSDT5Y2022.B25064-Data.csv', 2022)[['CountyFIPS', 'county', 'year', 'B25064_001E']]
rent_2023 = load_census_file('median_rent/ACSDT5Y2023.B25064-Data.csv', 2023)[['CountyFIPS', 'county', 'year', 'B25064_001E']]
rent_2024 = load_census_file('median_rent/ACSDT5Y2024.B25064-Data.csv', 2024)[['CountyFIPS', 'county', 'year', 'B25064_001E']]
df_rent = pd.concat([rent_2022, rent_2023, rent_2024], ignore_index=True)
df_rent = df_rent.rename(columns={'B25064_001E': 'median_rent'})
df_rent['median_rent'] = pd.to_numeric(df_rent['median_rent'], errors='coerce')
print("Rent:", df_rent.shape)
df_rent.head()

Rent: (186, 4)


,CountyFIPS,county,year,median_rent
0,36001,Albany County,2022,1196
1,36003,Allegany County,2022,735
2,36005,Bronx County,2022,1401
3,36007,Broome County,2022,882
4,36009,Cattaraugus County,2022,745


In [108]:
# B01001 - Age (for pct_age_25_34: male 25-29, 30-34 + female 25-29, 30-34)
age_cols = ['GEO_ID', 'NAME', 'B01001_001E', 'B01001_011E', 'B01001_012E', 'B01001_027E', 'B01001_028E']
dfs_age = []
for year, fname in [(2022, 'ACSDT5Y2022.B01001-Data.csv'), (2023, 'ACSDT5Y2023.B01001-Data.csv'), (2024, 'ACSDT5Y2024.B01001-Data.csv')]:
    df = pd.read_csv(f'median_sex_by_age/{fname}', skiprows=[1])[age_cols]
    df['CountyFIPS'] = df['GEO_ID'].str[-5:]
    df['county'] = df['NAME'].str.replace(', New York', '')
    df['year'] = year
    for c in ['B01001_001E', 'B01001_011E', 'B01001_012E', 'B01001_027E', 'B01001_028E']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['pct_age_25_34'] = (df['B01001_011E'] + df['B01001_012E'] + df['B01001_027E'] + df['B01001_028E']) / df['B01001_001E']
    dfs_age.append(df[['CountyFIPS', 'county', 'year', 'pct_age_25_34']])
df_age = pd.concat(dfs_age, ignore_index=True)
print("Age:", df_age.shape)
df_age.head()

Age: (186, 4)


,CountyFIPS,county,year,pct_age_25_34
0,36001,Albany County,2022,0.113395
1,36003,Allegany County,2022,0.095295
2,36005,Bronx County,2022,0.140095
3,36007,Broome County,2022,0.103476
4,36009,Cattaraugus County,2022,0.112961


In [109]:
# B15003 - Education (for pct_bachelors_plus: bachelor's + master's + professional + doctorate)
edu_cols = ['GEO_ID', 'NAME', 'B15003_001E', 'B15003_022E', 'B15003_023E', 'B15003_024E', 'B15003_025E']
dfs_edu = []
for year, fname in [(2022, 'ACSDT5Y2022.B15003-Data.csv'), (2023, 'ACSDT5Y2023.B15003-Data.csv'), (2024, 'ACSDT5Y2024.B15003-Data.csv')]:
    df = pd.read_csv(f'education_data/{fname}', skiprows=[1])[edu_cols]
    df['CountyFIPS'] = df['GEO_ID'].str[-5:]
    df['county'] = df['NAME'].str.replace(', New York', '')
    df['year'] = year
    for c in edu_cols[2:]:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['pct_bachelors_plus'] = (df['B15003_022E'] + df['B15003_023E'] + df['B15003_024E'] + df['B15003_025E']) / df['B15003_001E']
    dfs_edu.append(df[['CountyFIPS', 'county', 'year', 'pct_bachelors_plus']])
df_edu = pd.concat(dfs_edu, ignore_index=True)
print("Education:", df_edu.shape)
df_edu.head()

Education: (186, 4)


,CountyFIPS,county,year,pct_bachelors_plus
0,36001,Albany County,2022,0.444160
1,36003,Allegany County,2022,0.245036
2,36005,Bronx County,2022,0.212153
3,36007,Broome County,2022,0.298325
4,36009,Cattaraugus County,2022,0.202003


## 8. Merge Census Tables & Normalize County Names

Combine all Census tables. Normalize "St. Lawrence" → "Saint Lawrence" to match Zillow.

In [110]:
# Merge Census tables on CountyFIPS, county, year
df_census = df_income.merge(df_rent, on=['CountyFIPS', 'county', 'year'], how='outer')
df_census = df_census.merge(df_age, on=['CountyFIPS', 'county', 'year'], how='outer')
df_census = df_census.merge(df_edu, on=['CountyFIPS', 'county', 'year'], how='outer')

# Normalize county names to match Zillow (St. Lawrence → Saint Lawrence)
df_census['county'] = df_census['county'].str.replace('St. Lawrence County', 'Saint Lawrence County')

print(f"Census cleaned shape: {df_census.shape}")
print(f"Counties: {df_census['county'].nunique()}")
df_census.head(10)

Census cleaned shape: (186, 7)
Counties: 62


,CountyFIPS,county,year,median_income,median_rent,pct_age_25_34,pct_bachelors_plus
0,36001,Albany County,2022,78829,1196,0.113395,0.444160
1,36001,Albany County,2023,83149,1252,0.113700,0.451535
2,36001,Albany County,2024,85333,1313,0.112984,0.464216
3,36003,Allegany County,2022,58725,735,0.095295,0.245036
4,36003,Allegany County,2023,61233,754,0.096136,0.245619
5,36003,Allegany County,2024,62869,763,0.096291,0.248570
6,36005,Bronx County,2022,47036,1401,0.140095,0.212153
7,36005,Bronx County,2023,49036,1436,0.137292,0.219561
8,36005,Bronx County,2024,48676,1458,0.135342,0.223885
9,36007,Broome County,2022,58317,882,0.103476,0.298325


## 9. Save Census Cleaned Data

In [111]:
df_census.to_csv('census_cleaned.csv', index=False)
print(f"Saved census_cleaned.csv ({len(df_census):,} rows)")
print(f"Columns: {list(df_census.columns)}")

Saved census_cleaned.csv (186 rows)
Columns: ['CountyFIPS', 'county', 'year', 'median_income', 'median_rent', 'pct_age_25_34', 'pct_bachelors_plus']


## Part 2 Summary

**census_cleaned.csv** contains:
- `county`, `year`, `CountyFIPS`
- `median_income` (B19013)
- `median_rent` (B25064)
- `pct_age_25_34` (B01001)
- `pct_bachelors_plus` (B15003)

Ready to merge with `zillow_county_yearly.csv` for modeling.